# CAROTS on Colab - Step 1 (robustness) + Step 2 (variable-level localization)

This notebook trains and evaluates CAROTS on a GPU. The CAROTS model hard-codes `.cuda()`, so a GPU runtime is **required** (Runtime -> Change runtime type -> GPU).

It runs the whole pipeline:
1. Bring the **modified** CAROTS repository onto the runtime.
2. Install dependencies.
3. Generate the four VAR data variants (baseline + 3 flawed-training-data scenarios).
4. Generate and execute the run scripts (4 scenarios x 4 anomaly types).
5. Aggregate the metrics and render the Step 2 localization figures.
6. Copy everything back to Google Drive.

**Before you start:** upload the modified `CAROTS/` folder (the one in this workspace, including the `experiments/` package) to your Google Drive at `MyDrive/CAROTS`. The notebook copies it from there so your local edits are used instead of the unmodified upstream repo.

## 1. Check the GPU

In [6]:
!nvidia-smi

Sun Jul  5 17:25:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   76C    P0             38W /   70W |     425MiB /  15360MiB |     40%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Mount Google Drive

In [7]:
!pwd && ls
import torch; print("cuda:", torch.cuda.is_available())

/content
CAROTS	drive  sample_data
cuda: True


In [8]:
from google.colab import drive
drive.mount('/content/drive')
# from google.colab import drive  
# drive._mount('/content/drive') 

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Copy the modified repository onto the runtime

Adjust `DRIVE_REPO` if you uploaded the folder somewhere else.
(~3mins)

In [9]:
import os, shutil
DRIVE_REPO = '/content/drive/MyDrive/CAROTS'   # where you uploaded the modified repo
RUNTIME_REPO = '/content/CAROTS'
assert os.path.isdir(DRIVE_REPO), f'Upload the modified CAROTS folder to {DRIVE_REPO} first.'
if os.path.isdir(RUNTIME_REPO):
    shutil.rmtree(RUNTIME_REPO)
shutil.copytree(DRIVE_REPO, RUNTIME_REPO)
%cd /content/CAROTS
print('repo ready at', RUNTIME_REPO)

/content/CAROTS
repo ready at /content/CAROTS


## 4. Install dependencies

Colab already ships torch (CUDA), numpy, scipy, scikit-learn, pandas, matplotlib and tqdm. We add the extras CAROTS needs: `yacs`, `einops`, `torch_geometric` and `reformer_pytorch`. These last two are imported at module-load time by `models/carots/encoder.py` and `layers/SelfAttention_Family.py`, so they are required even for the default LSTM encoder.


In [10]:
!pip install -q yacs einops torch_geometric reformer_pytorch
import torch
from torch_geometric.nn import GATv2Conv          # used by models/carots/encoder.py
from reformer_pytorch import LSHSelfAttention     # used by layers/SelfAttention_Family.py
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())
print('torch_geometric + reformer_pytorch OK')


torch 2.11.0+cu128 cuda available: True
torch_geometric + reformer_pytorch OK


## 5. Generate the data variants

This writes `data/VAR_baseline/`, `data/VAR_nocausal/`, `data/VAR_nonstationary/` and `data/VAR_contaminated/`. Data generation is pure NumPy (no GPU needed) and is deterministic given the seed, so the datasets are reproducible.

In [11]:
!python -m experiments.datagen --variant all

[baseline] generating into /content/CAROTS/data/VAR_baseline (seed=0) ...
Nonstationary, beta=[[3. 0. 0. ... 0. 0. 1.]
 [0. 3. 0. ... 0. 0. 0.]
 [0. 0. 3. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 3. 1. 0.]
 [0. 0. 1. ... 0. 3. 0.]
 [0. 0. 0. ... 0. 0. 3.]], max_eig=27.9988
Nonstationary, beta=[[0.0750033 0.        0.        ... 0.        0.        0.0250011]
 [0.        0.0750033 0.        ... 0.        0.        0.       ]
 [0.        0.        0.0750033 ... 0.        0.        0.       ]
 ...
 [0.        0.        0.        ... 0.0750033 0.0250011 0.       ]
 [0.        0.        0.0250011 ... 0.        0.0750033 0.       ]
 [0.        0.        0.        ... 0.        0.        0.0750033]], max_eig=1.4566
Nonstationary, beta=[[0.0360442  0.         0.         ... 0.         0.         0.01201473]
 [0.         0.0360442  0.         ... 0.         0.         0.        ]
 [0.         0.         0.0360442  ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 0.0360

## 6. Generate the run scripts

In [12]:
!python -m experiments.run_experiments

wrote /content/CAROTS/experiments/generated/run_baseline.sh (4 runs)
wrote /content/CAROTS/experiments/generated/run_nocausal.sh (4 runs)
wrote /content/CAROTS/experiments/generated/run_nonstationary.sh (4 runs)
wrote /content/CAROTS/experiments/generated/run_contaminated.sh (4 runs)
wrote /content/CAROTS/experiments/generated/run_all.sh and commands.txt (16 total runs)


## 7. Sanity check: run the clean baseline first

Confirm the AUROC is high (near the paper's VAR numbers) before launching the full grid. This trains the causal discoverer, then the contrastive model, then evaluates on the four anomaly types.

In [13]:
!bash experiments/generated/run_baseline.sh

=== baseline ===
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12

## 8. Run all scenarios

This runs every (scenario x anomaly type) combination. It is the long-running cell. Each run saves the Step 2 per-variable artifacts because `TEST.SAVE_PER_VARIABLE=True` is set by the generator.

In [ ]:
!bash experiments/generated/run_all.sh

=== baseline ===
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12

## 9. Aggregate metrics + render localization figures

Writes `results/summary.csv`, the `results/comparison_*.png` charts and the per-variable localization figures under `results/localization/`.

In [ ]:
!python -m experiments.aggregate

/usr/bin/python3: Error while finding module specification for 'experiments.aggregate' (ModuleNotFoundError: No module named 'experiments')


In [ ]:
import pandas as pd
pd.read_csv('results/summary.csv').drop(columns=['result_dir'])

FileNotFoundError: [Errno 2] No such file or directory: 'results/summary.csv'

## 10. Copy results back to Google Drive

In [ ]:
import shutil, os
DEST = '/content/drive/MyDrive/CAROTS_results'
if os.path.isdir(DEST):
    shutil.rmtree(DEST)
shutil.copytree('results', DEST)
print('results copied to', DEST)

FileNotFoundError: [Errno 2] No such file or directory: 'results'